## 3.4 何时写入记忆

官方介绍了两种方式。
1. 在主流程里写（hot path）
2. 在后台写（background）

**核对：官方确实只分了两种**

LangChain 官方文档 Memory 概念页的 “Writing memories” 一节只列了两种写入时机：**In the hot path**（在主流程里写）和 **In the background**（在后台写）。两者的区别在于记忆在什么时候写、由谁决定写什么：

| | 方式一：在主流程里写（hot path） | 方式二：在后台写（background） |
|---|---|---|
| 什么时候写 | 回复用户**之前**，在 Agent 运行过程中 | 回复用户**之后**，另起一个任务 |
| 谁决定写什么 | Agent 自己，通常通过一个“保存记忆”工具 | 单独的提取逻辑，通常是另一次模型调用 |
| 优点 | 实时：写完马上能用；可以告诉用户“已记住” | 不增加回复延迟；记忆逻辑和业务逻辑分开，Agent 专心回答 |
| 缺点 | 增加延迟；要多一个工具；模型要一边回答一边判断记什么，记忆的数量和质量会受影响 | 写完之前其他会话读不到新记忆；要决定多久写一次、什么时候触发 |

后台写入的触发方式，官方列了三种：一段时间没有新消息后再写（有新消息就重新计时）、按 cron 定时写、手动触发。下面方式二的例子是在每次回复后立即手动触发。

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-6-sol",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

两个例子共用的代码：同一句用户输入，用 `CustomState` 传入 user_id（同 3.3.1），以及两个打印函数。中转模型返回的 `AIMessage.content` 是内容块列表，直接 `pretty_print` 会打印出原始内容块，所以用 `show_messages` 只打印文本和工具调用。

In [ ]:
import time
import asyncio
from typing import NotRequired
from uuid import uuid4

from langchain.agents import AgentState
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.prebuilt import ToolRuntime
from langgraph.store.memory import InMemoryStore

USER_INPUT = "我叫小花，最喜欢喝紫光园的奶皮子酸奶，不吃香菜。帮我推荐一道晚饭吧。"


class CustomState(AgentState):
    user_id: NotRequired[str]


def show_memories(store, user_id: str) -> None:
    items = store.search(("users", user_id, "memories"))
    print(f"store 里有 {len(items)} 条记忆")
    for item in items:
        print("  -", item.value["content"])

def show_messages(messages) -> None:
    """只打印消息文本和工具调用（中转模型的 content 是内容块列表，直接 pretty_print 不好读）"""
    for msg in messages:
        if msg.text:
            print(f"[{msg.type}] {msg.text}")
        for call in getattr(msg, "tool_calls", []):
            print(f"[{msg.type}] 调用工具 {call['name']}({call['args']})")

**方式一：在主流程里写（hot path）**

做法就是 3.3.1 的“在工具中访问长期记忆”：给 Agent 一个 `save_memory` 工具，让模型在回答前自己决定要不要保存、保存什么。

1. 定义 `save_memory` 工具：参数 `content` 由模型填写，`runtime` 由 tools 节点注入（见 3.3.1 第三、四步）。工具用 `runtime.state["user_id"]` 组成 namespace，写入 store。
2. 在 `system_prompt` 里要求：用户透露个人信息或偏好时，先保存再回复。
3. invoke 后的执行顺序：模型调用 → tools 节点写入 store → 模型再调用一次，生成回复。

key 用 `uuid4()` 生成，每条记忆一个随机 ID，多条记忆不会互相覆盖；namespace 的最后一段 `"memories"` 用来和其他类型的数据区分。

下面这个单元格会调用中转模型 2 次：

In [ ]:
hot_store = InMemoryStore()


@tool(parse_docstring=True)
def save_memory(content: str, runtime: ToolRuntime) -> str:
    """把用户透露的个人信息或长期偏好保存为长期记忆。

    Args:
        content: 一条简短的事实，例如“用户不吃香菜”
    """
    namespace = ("users", runtime.state["user_id"], "memories")
    runtime.store.put(namespace, str(uuid4()), {"content": content})
    return "saved"


hot_agent = create_agent(
    model=model,
    tools=[save_memory],
    store=hot_store,
    state_schema=CustomState,
    system_prompt="你是一个贴心的助手。用户透露个人信息或长期偏好时，先调用 save_memory 保存（每条信息调用一次），再回复用户。",
)

start = time.perf_counter()
response = hot_agent.invoke({"messages": [HumanMessage(USER_INPUT)], "user_id": "user-1"})
elapsed = time.perf_counter() - start

show_messages(response["messages"])

print("=" * 60)
print(f"invoke 耗时 {elapsed:.1f} 秒，调用模型 {sum(isinstance(m, AIMessage) for m in response['messages'])} 次")
print("invoke 返回时：", end="")
show_memories(hot_store, "user-1")

[human] 我叫小花，最喜欢喝紫光园的奶皮子酸奶，不吃香菜。帮我推荐一道晚饭吧。
[ai] 调用工具 save_memory({'content': '用户叫小花。'})
[ai] 调用工具 save_memory({'content': '用户最喜欢喝紫光园的奶皮子酸奶。'})
[ai] 调用工具 save_memory({'content': '用户不吃香菜。'})
[tool] saved
[tool] saved
[tool] saved
[ai] 小花，晚饭推荐你吃**番茄牛肉盖饭**：热乎、饱腹，番茄的酸甜也很开胃。点餐时记得备注**不要香菜**；饭后再喝一杯你喜欢的紫光园奶皮子酸奶，刚刚好。
invoke 耗时 11.0 秒，调用模型 2 次
invoke 返回时：store 里有 3 条记忆
  - 用户叫小花。
  - 用户最喜欢喝紫光园的奶皮子酸奶。
  - 用户不吃香菜。


从输出可以看到：

* 第一次模型调用一次返回了 3 个 `save_memory` 调用，把一句话拆成了 3 条记忆。
* 3 次写入都在 tools 节点里完成，之后第二次模型调用才生成回复。用户要等这两次模型调用都结束。
* `invoke` 返回时 store 里已经有 3 条记忆，下一次对话马上能用。
* 记什么、拆成几条，全由模型在回答问题的同时判断；换一个模型或 prompt，结果可能不同。

**方式二：在后台写（background）**

主流程的 Agent 不带任何记忆工具，只负责回答。拿到回复后，再启动一个后台任务，用另一次模型调用从对话里提取记忆，写入 store。

1. 定义提取格式 `Memories`，用 `model.with_structured_output(Memories)` 让模型直接返回记忆列表。
2. `write_memories` 是后台任务：把对话整理成一段文本交给模型提取，再用 `aput` 逐条写入。
3. 主流程用 `ainvoke` 拿到回复后，用 `asyncio.create_task(...)` 启动后台任务。

用到的异步写法：

* `async def` 定义的是协程函数，调用它不会立刻执行，而是得到一个协程对象；`await` 表示“等它执行完再往下走”。
* `ainvoke`、`aput` 分别是 `invoke`、`put` 的异步版本。
* `asyncio.create_task(协程)` 把协程交给事件循环在后台执行，自己立刻返回，不等它结束。
* Jupyter 本身运行在事件循环里，所以单元格里可以直接写 `await`。

因此 `create_task` 之后马上查看 store，后台任务还没开始执行，store 是空的；`await task` 等它结束后再看，才有记忆。实际应用（例如 Web 服务）里，主流程把回复发给用户后不必 `await` 这个任务。

下面这个单元格会调用中转模型 2 次（一次回答，一次提取记忆）：

In [ ]:
bg_store = InMemoryStore()


class Memories(BaseModel):
    memories: list[str] = Field(description="值得长期记住的用户信息，每条一句话；没有就返回空列表")


extractor = model.with_structured_output(Memories)


async def write_memories(messages, user_id: str) -> None:
    """后台任务：用模型从对话中提取长期记忆，再写入 store"""
    start = time.perf_counter()
    conversation = "\n".join(f"{m.type}: {m.text}" for m in messages)
    result = await extractor.ainvoke([
        SystemMessage("从下面的对话中提取关于用户的长期信息（身份、偏好、习惯），忽略一次性的请求。"),
        HumanMessage(conversation),
    ])
    namespace = ("users", user_id, "memories")
    for content in result.memories:
        await bg_store.aput(namespace, str(uuid4()), {"content": content})
    print(f"[后台] 提取并写入 {len(result.memories)} 条记忆，耗时 {time.perf_counter() - start:.1f} 秒")


# 主流程的 Agent 不带记忆工具，只负责回答
bg_agent = create_agent(model=model, system_prompt="你是一个贴心的助手。")

start = time.perf_counter()
response = await bg_agent.ainvoke({"messages": [HumanMessage(USER_INPUT)]})
elapsed = time.perf_counter() - start

show_messages(response["messages"])

print("=" * 60)
print(f"ainvoke 耗时 {elapsed:.1f} 秒，调用模型 {sum(isinstance(m, AIMessage) for m in response['messages'])} 次")

# 回复已经拿到，再启动后台任务写记忆
task = asyncio.create_task(write_memories(response["messages"], "user-1"))
print("回复返回、后台任务刚启动时：", end="")
show_memories(bg_store, "user-1")

await task  # 演示用：等后台任务结束再查看；实际应用里主流程不必等它
print("后台任务结束后：", end="")
show_memories(bg_store, "user-1")

[human] 我叫小花，最喜欢喝紫光园的奶皮子酸奶，不吃香菜。帮我推荐一道晚饭吧。
[ai] 小花，今晚推荐你吃**番茄牛腩盖饭**：酸甜开胃、热乎又顶饱。点餐时记得备注**不要香菜**。饭后再来一杯你喜欢的紫光园奶皮子酸奶，刚刚好！
ainvoke 耗时 7.8 秒，调用模型 1 次
回复返回、后台任务刚启动时：store 里有 0 条记忆


[后台] 提取并写入 3 条记忆，耗时 7.0 秒
后台任务结束后：store 里有 3 条记忆
  - 用户叫小花。
  - 用户最喜欢喝紫光园的奶皮子酸奶。
  - 用户不吃香菜。


从输出可以看到：

* 主流程只调用了 1 次模型。回复里同样用上了“不要香菜”“奶皮子酸奶”，这是因为这些信息就在本轮消息里，和长期记忆无关。
* 回复返回时 store 里是 0 条，后台任务又花了约 7 秒才写入 3 条。这段时间里如果有别的会话来读，读不到这些记忆。
* 提取 prompt 只做一件事（提取长期信息，忽略一次性请求），所以“推荐晚饭”没有被记下来。记忆的质量取决于这个提取 prompt，和主流程的 prompt 无关。
* 这个例子每次回复后都立即提取。官方列出的另一种触发方式是等对话停下来一段时间再提取（期间有新消息就重新计时），避免对同一段对话反复提取。

**两种方式对比**

![两种写入时机](assets/15-memory-write-timing.svg)

| | 方式一：主流程里写 | 方式二：后台写 |
|---|---|---|
| 主流程的模型调用 | 2 次（决定保存 + 生成回复） | 1 次（生成回复） |
| 本次实测：用户等待 | 11.0 秒 | 7.8 秒 |
| 记忆什么时候可用 | 回复前已写好 | 回复后约 7 秒，后台任务结束时 |
| 谁决定记什么 | 主 Agent，在回答问题的同时判断 | 单独的提取 prompt |
| 总的模型调用 | 2 次 | 2 次，其中一次挪到了后台 |

单次运行的耗时受网络和模型波动影响，只作参考；稳定的差别是用户等待期间的模型调用次数。